# Efficient FunSearch - Demo

Sample-efficient FunSearch with duplicate code detection.

**Course**: CS5491 Deep Learning Project

---

## 1. Setup

### 1.1 Check GPU

In [ ]:
!nvidia-smi

### 1.2 Clone Repository

Replace `YOUR_USERNAME/YOUR_REPO` with your GitHub repository.

In [ ]:
# Option A: Clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO

# Option B: Upload files directly (for development)
# from google.colab import files
# uploaded = files.upload()

### 1.3 Install Dependencies

In [ ]:
# Install core dependencies
!pip install -q numpy sentence-transformers torch

# Install the package
!pip install -e .

### 1.4 Import Modules

In [ ]:
import sys

sys.path.insert(0, '.')

from src import (
    EfficiencyTracker,
    HybridSimilarityDetector,
    ProgramArchive,
    ProgramNormalizer,
)

print("All modules imported successfully!")

---

## 2. Core Functionality Demo

### 2.1 Code Normalization

In [ ]:
# Create normalizer
normalizer = ProgramNormalizer()

# Example: Two functionally equivalent programs
code_a = '''
def solve(items, capacity):
    """A knapsack solver."""
    result = 0
    for item in items:
        if item <= capacity:
            result += item
    return result
'''

code_b = '''
def solve(xs, cap):
    total = 0
    for x in xs:
        if x <= cap:
            total += x
    return total
'''

# Normalize both
norm_a = normalizer.normalize(code_a)
norm_b = normalizer.normalize(code_b)

print("=== Original Code A ===")
print(code_a)
print("\n=== Normalized Code A ===")
print(norm_a.canonical_code)
print(f"AST Hash: {norm_a.ast_hash[:16]}...")

print("\n=== Normalized Code B ===")
print(norm_b.canonical_code)
print(f"AST Hash: {norm_b.ast_hash[:16]}...")

print(f"\nSame AST hash? {norm_a.ast_hash == norm_b.ast_hash}")

### 2.2 Similarity Detection

In [ ]:
# Create detector
detector = HybridSimilarityDetector()

# Check similarity
result = detector.is_similar(norm_a, norm_b)

print(f"Is duplicate? {result.is_duplicate}")
print(f"Detection method: {result.detection_method}")
print(f"AST similarity: {result.ast_similarity:.4f}")
if result.embedding_similarity is not None:
    print(f"Embedding similarity: {result.embedding_similarity:.4f}")

### 2.3 Program Archive

In [ ]:
# Create archive
archive = ProgramArchive()

# Add programs
id_a = archive.add(code_a, norm_a, score=0.85)
print(f"Added program A with ID: {id_a[:8]}...")

# Try to add duplicate
id_b = archive.add(code_b, norm_b, score=0.90)
print(f"Added program B with ID: {id_b[:8]}... (should be None if duplicate)")

# Check duplicate detection
is_dup = archive.is_duplicate(norm_b)
print(f"\nIs B a duplicate? {is_dup}")

# Archive stats
stats = archive.get_stats()
print("\nArchive stats:")
print(f"  Total programs: {stats['total_programs']}")
print(f"  Duplicates skipped: {stats['duplicates_skipped']}")

### 2.4 Efficiency Tracking

In [ ]:
# Create tracker
tracker = EfficiencyTracker()

# Simulate some operations
for i in range(100):
    tracker.record_generation()
    if i % 3 == 0:  # 33% are duplicates
        tracker.record_duplicate()
    tracker.record_evaluation()

# Get metrics
metrics = tracker.get_metrics()
print("Efficiency Metrics:")
print(f"  Total generated: {metrics.total_generated}")
print(f"  Duplicates found: {metrics.duplicates_found}")
print(f"  Unique programs: {metrics.unique_programs}")
print(f"  Duplicate rate: {metrics.duplicate_rate:.1%}")
print(f"  Estimated savings: {metrics.estimated_llm_calls_saved} LLM calls")

---

## 3. FunSearch Integration Demo

### 3.1 Install FunSearch

In [ ]:
# Install FunSearch from community fork
!pip install -q git+https://github.com/RayZhhh/funsearch.git

### 3.2 Run FunSearch with Duplicate Detection

This section demonstrates how to integrate our duplicate detection with FunSearch.

In [ ]:
# TODO: Import FunSearch adapter after T033 is implemented
# from src.integration import FunSearchAdapter

# Example usage (pseudocode):
# adapter = FunSearchAdapter(
#     problem="tsp",
#     max_generations=100,
#     use_duplicate_detection=True
# )
# result = adapter.run()
# print(f"Best score: {result.best_score}")
# print(f"LLM calls saved: {result.llm_calls_saved}")

---

## 4. Run Tests

In [ ]:
# Install pytest
!pip install -q pytest

# Run unit tests
!pytest tests/unit -v --tb=short

---

## 5. Results & Analysis

### 5.1 Compare: With vs Without Duplicate Detection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example comparison data (replace with actual experiment results)
generations = np.arange(0, 100)

# Simulated efficiency improvement
unique_programs_baseline = generations + 1
unique_programs_efficient = np.where(
    generations < 20,
    generations + 1,
    20 + (generations - 20) * 0.7  # 30% duplicates filtered
)

plt.figure(figsize=(10, 6))
plt.plot(generations, unique_programs_baseline, label='Baseline FunSearch', linestyle='--')
plt.plot(generations, unique_programs_efficient, label='Efficient FunSearch (ours)', linewidth=2)
plt.xlabel('Generation')
plt.ylabel('Unique Programs Evaluated')
plt.title('Sample Efficiency Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Calculate savings
savings = (unique_programs_baseline[-1] - unique_programs_efficient[-1]) / unique_programs_baseline[-1] * 100
print(f"Estimated LLM call reduction: {savings:.1f}%")

---

## 6. Summary

In [ ]:
print("""
=== Efficient FunSearch Demo Complete ===

Key Features:
1. Code Normalization: Standardizes variable names, removes docstrings
2. Similarity Detection: Hybrid approach (embedding + AST)
3. Program Archive: Stores unique programs, skips duplicates
4. Efficiency Tracking: Measures improvement from duplicate detection

Next Steps:
- Run full FunSearch experiments
- Compare results with baseline
- Tune detection thresholds
- Prepare final report
""")